# TimelyMT Policy V2: Post-Hoc Exploratory DEV Extension

Before **Run All**: create the private Dataset `iteams24/timelymt-policy-v2-checkpoints` once if it does not exist; add the `KAGGLE_API_TOKEN` secret; enable Internet and a GPU. This runner versions that one V2 Dataset and stops before TEST.

## CONFIG

In [ ]:
from pathlib import Path
import json
REPOSITORY_URL = "https://github.com/MinhCYB/TimelyMT.git"
REPOSITORY_REF = "main"  # Set to the committed V2 branch/SHA before Run All.
V1_CHECKPOINT_DATASET_REF = "iteams24/timelymt-research-checkpoints"
V2_CHECKPOINT_DATASET_REF = "iteams24/timelymt-policy-v2-checkpoints"
MODEL_ID = "VietAI/envit5-translation"
MODEL_REVISION = "840bc88104d5a4277af740eaedb024df8c3093e7"
ENCODER_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
ENCODER_REVISION = "e62509716f15c5fd03a6fd3156a4bc5e43f83f26"
THRESHOLDS = ["0.30", "0.40", "0.50", "0.60", "0.70"]
RUN_EMERGENCY_CHECKPOINT = False
WORKING = Path("/kaggle/working")
REPO = WORKING / "TimelyMT"
DOWNLOADS = WORKING / "checkpoint-downloads"
EXPORT = WORKING / "timelymt-policy-v2-artifacts.tar.gz"

## CLONE CURRENT V2 REPOSITORY

In [ ]:
import subprocess
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, "--single-branch", REPOSITORY_URL, str(REPO)], check=True)
else:
    print("Repository already present; preserving resumable workspace")
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)

## SOURCE-TREE IMPORT BOOTSTRAP

In [ ]:
import os, sys
os.environ["PYTHONPATH"] = str(REPO / "src")
sys.path.insert(0, str(REPO / "src"))
from timelymt.research.policy_v2 import EXPERIMENT_STATUS, restore_v1_artifacts
assert EXPERIMENT_STATUS == "post_hoc_exploratory"

## ENVIRONMENT + CUDA PRECHECK

In [ ]:
import importlib.util, torch
assert torch.cuda.is_available(), "Policy V2 Kaggle execution requires CUDA"
properties = torch.cuda.get_device_properties(0)
print({"gpu_name": properties.name, "cuda_capability": torch.cuda.get_device_capability(0), "vram_gib": properties.total_memory / 2**30})
missing = [name for name in ("transformers", "sacrebleu", "sklearn") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
# PyTorch is intentionally never reinstalled.
import transformers
assert int(transformers.__version__.split('.')[0]) == 4

## KAGGLE AUTH

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
if not token:
    raise RuntimeError("KAGGLE_API_TOKEN secret is required")
os.environ["KAGGLE_API_TOKEN"] = token
subprocess.run(["kaggle", "--version"], check=True)  # Use the Kaggle console executable directly.
DOWNLOADS.mkdir(parents=True, exist_ok=True)
import shutil, tarfile
def publish_v2_boundary(stage):
    boundary_path = REPO / "outputs/experiments/policy-v2/checkpoint-metadata.json"
    boundary_path.parent.mkdir(parents=True, exist_ok=True)
    boundary_path.write_text(json.dumps({"checkpoint_stage": stage, "experiment_status": "post_hoc_exploratory", "v1_source_commit": "6c75da5d60cc626ab79e7e82cae471e18be27531", "v2_code_commit": subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()}, indent=2) + "\n")
    with tarfile.open(EXPORT, "w:gz") as archive:
        for relative in (Path("checkpoints/policy_v2"), Path("outputs/experiments/policy-v2")):
            source_root = REPO / relative
            if not source_root.exists():
                continue
            for path in sorted(source_root.rglob("*")):
                rel = path.relative_to(REPO)
                lowered = rel.as_posix().lower()
                if path.is_file() and "embedding-cache" not in rel.parts and "pseudo_labels" not in rel.parts and "/test/" not in f"/{lowered}/":
                    archive.add(path, arcname=rel.as_posix(), recursive=False)
        archive.add(REPO / "configs/experiments/policy-v2.json", arcname="outputs/experiments/policy-v2/policy-v2-config.json", recursive=False)
    upload = WORKING / "v2-dataset-upload"
    upload.mkdir(exist_ok=True)
    shutil.copy2(EXPORT, upload / EXPORT.name)
    metadata = {"title": "TimelyMT Policy V2 Checkpoints", "id": V2_CHECKPOINT_DATASET_REF, "licenses": [{"name": "other"}]}
    (upload / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
    result = subprocess.run(["kaggle", "datasets", "version", "-p", str(upload), "-m", stage, "--dir-mode", "zip"], check=False)
    if result.returncode != 0:
        raise RuntimeError("Create the private V2 Dataset once before Run All; boundary version upload failed")
    print(f"persisted boundary={stage}")

## DOWNLOAD / VALIDATE IMMUTABLE V1 INPUT ARTIFACT

In [ ]:
v1_download = DOWNLOADS / "v1"
v1_download.mkdir(parents=True, exist_ok=True)
subprocess.run(["kaggle", "datasets", "download", "-d", V1_CHECKPOINT_DATASET_REF, "-p", str(v1_download), "--unzip"], check=True)
raw_candidates = list(v1_download.rglob("timelymt-checkpoint.tar.gz"))
expanded_candidates = [path for path in v1_download.rglob("timelymt-checkpoint") if path.is_dir()]
v1_source = raw_candidates[0] if raw_candidates else expanded_candidates[0] if expanded_candidates else v1_download
identity = restore_v1_artifacts(v1_source, REPO)
print(identity)

## RESTORE VALID V2 RESUME ARTIFACTS

In [ ]:
v2_download = DOWNLOADS / "v2"
v2_download.mkdir(parents=True, exist_ok=True)
result = subprocess.run(["kaggle", "datasets", "download", "-d", V2_CHECKPOINT_DATASET_REF, "-p", str(v2_download), "--unzip"], check=False)
if result.returncode == 0:
    archives = list(v2_download.rglob("timelymt-policy-v2-artifacts.tar.gz"))
    expanded = [path for path in v2_download.rglob("timelymt-policy-v2-artifacts") if path.is_dir()]
    if archives:
        with tarfile.open(archives[0], "r:gz") as archive:
            for member in archive.getmembers():
                parts = Path(member.name.replace('\\', '/')).parts
                if member.issym() or member.islnk() or '..' in parts or not (member.isfile() or member.isdir()):
                    raise RuntimeError(f"unsafe V2 resume member: {member.name}")
                if parts and parts[0] not in {"checkpoints", "outputs"}:
                    raise RuntimeError(f"unexpected V2 resume member: {member.name}")
            archive.extractall(REPO, filter="data")
    elif expanded:
        for relative in (Path("checkpoints/policy_v2"), Path("outputs/experiments/policy-v2")):
            source = expanded[0] / relative
            if source.exists():
                for path in source.rglob("*"):
                    if path.is_symlink():
                        raise RuntimeError(f"symlink forbidden in expanded V2 checkpoint: {path}")
                    if path.is_file():
                        destination = REPO / relative / path.relative_to(source)
                        destination.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(path, destination)
else:
    print("No existing V2 Dataset version; starting from immutable V1 inputs")

## AUDIT V1 TRAIN/DEV SUPERVISION

In [ ]:
from timelymt.research.policy_v2 import validate_v1_supervision
train_manifest, train_rows = validate_v1_supervision(REPO / "data/policy/pseudo_labels/train", "train")
dev_manifest, dev_rows = validate_v1_supervision(REPO / "data/policy/pseudo_labels/dev", "dev")
print({"train_states": len(train_rows), "dev_states": len(dev_rows), "train_labels": {k: train_manifest[k] for k in ("LISTEN", "COMMIT")}})
del train_rows, dev_rows  # Training commands reload immutable persisted records; no EnViT5 regeneration.

## DOWNLOAD / CACHE FROZEN MINILM ENCODER

In [ ]:
from transformers import AutoModel, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL_ID, revision=ENCODER_REVISION)
encoder = AutoModel.from_pretrained(ENCODER_MODEL_ID, revision=ENCODER_REVISION, torch_dtype=torch.float16).eval().requires_grad_(False).cuda()
assert encoder.config.hidden_size == 384
del encoder, tokenizer
torch.cuda.empty_cache()

## PRECOMPUTE TRAIN EMBEDDINGS

In [ ]:
from timelymt.research.policy_v2 import EmbeddingCache, FrozenMiniLMEncoder
_, train_rows = validate_v1_supervision(REPO / "data/policy/pseudo_labels/train", "train")
cache = EmbeddingCache(REPO / "outputs/experiments/policy-v2/embedding-cache", FrozenMiniLMEncoder(batch_size=256))
unique_texts = sorted({row["causal"][field] for row in train_rows for field in ("current_source_text", "previous_committed_source_text", "previous_committed_target_text") if row["causal"][field]})
for start in range(0, len(unique_texts), 2048):
    cache.encode(unique_texts[start:start + 2048])
print({"unique_train_texts_cached": len(unique_texts)})
del train_rows, unique_texts, cache
torch.cuda.empty_cache()

## TRAIN V2-P0

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "train-v2", "--variant", "P0"], cwd=REPO, check=True)

## TRAIN V2-P1

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "train-v2", "--variant", "P1"], cwd=REPO, check=True)

## TRAIN V2-P2

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "train-v2", "--variant", "P2"], cwd=REPO, check=True)

## CHECKPOINT V2 MODELS

In [ ]:
from timelymt.research.policy_v2_runner import _valid_checkpoint
assert all(_valid_checkpoint(variant) for variant in ("P0", "P1", "P2"))
publish_v2_boundary("v2-models-trained")

## DEV V2 ROLLOUTS

In [ ]:
for variant in ("P0", "P1", "P2"):
    subprocess.run([sys.executable, "-m", "timelymt.research.cli", "rollout-v2", "--split", "dev", "--variant", variant, "--thresholds", *THRESHOLDS, "--batch-size", "16"], cwd=REPO, check=True)

## CHECKPOINT DEV PREDICTIONS

In [ ]:
from timelymt.research.policy_v2_runner import all_v2_strategies
dev_ids = set(json.loads((REPO / "data/splits/experimental.json").read_text())["splits"]["dev"])
for strategy in all_v2_strategies():
    assert {path.stem for path in (REPO / "outputs/experiments/policy-v2/predictions/dev" / strategy).glob("*.json")} == dev_ids
publish_v2_boundary("v2-dev-rollouts-complete")

## V2 EVALUATION

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "evaluate-v2", "--split", "dev"], cwd=REPO, check=True)

## V1/V2 COMPARISON

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "compare-v2"], cwd=REPO, check=True)

## V2 DEV SELECTION

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "select-v2", "--split", "dev"], cwd=REPO, check=True)

## FREEZE V2 EXPLORATORY CONFIG

In [ ]:
subprocess.run([sys.executable, "-m", "timelymt.research.cli", "freeze-v2", "--split", "dev"], cwd=REPO, check=True)
frozen = json.loads((REPO / "outputs/experiments/policy-v2/v2-frozen-config.json").read_text())
assert frozen["artifact_status"] == "v2-dev-frozen-complete" and frozen["test_status"] == "UNTOUCHED"
print("boundary=v2-dev-frozen-complete")

## EXPORT + VERSION V2 DATASET

In [ ]:
import hashlib
publish_v2_boundary("v2-dev-frozen-complete")
with tarfile.open(EXPORT, "r:gz") as archive:
    names = archive.getnames()
assert names and not any("pseudo_labels" in name or "embedding-cache" in name or "huggingface" in name.lower() or "test" in name.lower() for name in names)
print({"export": str(EXPORT), "sha256": hashlib.sha256(EXPORT.read_bytes()).hexdigest(), "members": len(names)})
if RUN_EMERGENCY_CHECKPOINT:
    print("Manual emergency checkpoint requested; normal frozen boundary already uploaded")

# STOP BEFORE TEST

This post-hoc exploratory runner intentionally ends after full DEV evaluation, comparison, selection, freeze, and export. Do not add TEST commands to this notebook.